# VolleyTag — Danya detector v2
Train on annotated sets 1 and 2 only. The frozen third set is never used here.

In [ ]:
!nvidia-smi
!pip -q install ultralytics==8.4.192

In [ ]:
from pathlib import Path

input_root = Path('/kaggle/input')
print('mounted inputs:', [str(p) for p in input_root.iterdir()])
model_path = next(input_root.rglob('danya-detector-v1-best.pt'))
source_yaml = next(input_root.rglob('dataset.yaml'))
dataset_root = source_yaml.parent
data_yaml = Path('/kaggle/working/dataset.yaml')
data_yaml.write_text(
    f'path: {dataset_root}\n'
    'train: images/train\n'
    'val: images/val\n'
    'names:\n  0: danya\n'
)
train_count = len(list((dataset_root / 'images/train').glob('*')))
val_count = len(list((dataset_root / 'images/val').glob('*')))
print('dataset:', dataset_root)
print('model:', model_path)
print('train images:', train_count)
print('val images:', val_count)
assert train_count == 2190 and val_count == 315, (train_count, val_count)

In [ ]:
from ultralytics import YOLO

model = YOLO(str(model_path))
results = model.train(
    data=str(data_yaml),
    epochs=80,
    imgsz=960,
    batch=16,
    device=0,
    workers=4,
    patience=16,
    close_mosaic=10,
    cache='disk',
    seed=51,
    project='/kaggle/working/runs',
    name='danya-detector-v2',
    plots=True,
)

In [ ]:
from pathlib import Path
import json, shutil

run = Path('/kaggle/working/runs/danya-detector-v2')
best = run / 'weights/best.pt'
shutil.copy2(best, '/kaggle/working/danya-detector-v2-best.pt')
metrics = {
    'precision': float(results.results_dict.get('metrics/precision(B)', 0)),
    'recall': float(results.results_dict.get('metrics/recall(B)', 0)),
    'map50': float(results.results_dict.get('metrics/mAP50(B)', 0)),
    'map50_95': float(results.results_dict.get('metrics/mAP50-95(B)', 0)),
}
Path('/kaggle/working/validation-metrics.json').write_text(json.dumps(metrics, indent=2))
print(metrics)
print('best weights:', best)